# ZADANIE A

In [24]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

image_paths = ["./img/dog_grass.jpg", "./img/red_car.jpg", "./img/animal_in_the_house.jpg", "./img/burger.jpg", "./img/building.jpg"]
images = [Image.open(p).convert("RGB") for p in image_paths]

queries = [
    "dog on the grass",
    "animal in the house",
    "something to eat",
    "building in the city",
    "red car"
]

with torch.no_grad():
    inputs = processor(text=queries, images=images, return_tensors="pt", padding=True)
    outputs = model(**inputs)

text_emb = outputs.text_embeds.numpy()
image_emb = outputs.image_embeds.numpy()
sim = cosine_similarity(text_emb, image_emb)
for i, q in enumerate(queries):
    best = np.argmax(sim[i])
    print(f"Query: {q}")
    print(f"Best image: {image_paths[best]} (score={sim[i][best]:.3f})")
    print("-" * 60)


Query: dog on the grass
Best image: ./img/dog_grass.jpg (score=0.286)
------------------------------------------------------------
Query: animal in the house
Best image: ./img/animal_in_the_house.jpg (score=0.243)
------------------------------------------------------------
Query: something to eat
Best image: ./img/burger.jpg (score=0.227)
------------------------------------------------------------
Query: building in the city
Best image: ./img/building.jpg (score=0.235)
------------------------------------------------------------
Query: red car
Best image: ./img/red_car.jpg (score=0.242)
------------------------------------------------------------


* Model **poprawnie dopasował obrazy** do zapytań jednoznacznych:

  * `"dog on the grass"` → `dog_grass.jpg`
  * `"animal in the house"` → `animal_in_the_house.jpg`
  * `"red car"` → `red_car.jpg`
  * `"building in the city"` → `building.jpg`

* Zapytanie **niejednoznaczne** `"something to eat"` zostało dopasowane do `burger.jpg`, co jest sensowne, ale model może mieć trudności, jeśli w zbiorze byłyby inne jedzenie, np. owoce czy napoje – nie ma kontekstu i wybór zależy od tego, co jest w zbiorze i jak wytrenowany jest CLIP.

* Wyniki wskazują, że wartości cosine similarity są raczej niskie (~0.22–0.28), co jest normalne przy CLIP i niewielkiej liczbie obrazów, ale oznacza, że model nie jest bardzo pewny w tych dopasowaniach.



# ZADANIE B

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

model_name = "xlm-roberta-base" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2) 


e:\studies_master\MlShared\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matkan1\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For 

In [19]:
dataset = load_dataset("imdb") 
train_texts = list(dataset['train']['text'])
train_labels = list(dataset['train']['label'])

test_texts = list(dataset['test']['text'])
test_labels = list(dataset['test']['label'])

In [20]:
def preprocess(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True)
    encodings['labels'] = labels
    return encodings


train_enc = preprocess(train_texts, train_labels)
test_enc = preprocess(test_texts, test_labels)

In [21]:
from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

train_dataset = SentimentDataset(train_enc)
test_dataset = SentimentDataset(test_enc)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100,
    save_strategy="epoch",
    learning_rate=2e-5,
    seed=42,
    fp16=True if torch.cuda.is_available() else False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step,Training Loss
100,0.596900
200,0.602900
300,0.537100
400,0.528600
500,0.642200
600,0.516700
700,0.502000
800,0.464100
900,0.483000
1000,0.510800


TrainOutput(global_step=3125, training_loss=0.4563943911743164, metrics={'train_runtime': 286.2972, 'train_samples_per_second': 87.322, 'train_steps_per_second': 10.915, 'total_flos': 6577776384000000.0, 'train_loss': 0.4563943911743164, 'epoch': 1.0})

In [22]:
def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
    enc = enc.to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.argmax(logits, dim=-1).tolist()


english_samples = ["I loved this movie!", "It was terrible..."]
preds_eng = predict(english_samples)
print(preds_eng)  


[1, 0]


In [23]:
polish_samples = ["Bardzo podobał mi się ten film!", "Był okropny..."]
preds_pl = predict(polish_samples)
print(preds_pl)


[1, 0]
